# RQ2 

## Setup

* Follow the instructions on ```Readme.md``` 
* Inside the *pipeline* folder, run ```python -m src.rq2```. This will parse the .json file containing the dataset into  ```bigcodebench_clone_dataset.xml``` file containing all clone pairs (exclusing the original code). 
* Checkout [CloneCognition](https://github.com/pseudoPixels/CloneCognition)
* Paste the ```bigcodebench_clone_dataset.xml``` file inside ```input_clone_pairs/``` on the CloneCognition project.
    * Make sure to follow the CloneCognition instructions on their README
* Run ```python validateClones.py 0.76 input_clone_pairs/ out/```. This will produce a file  ```bigcodebench_clone_dataset.xml.mlValidated```
* The file ```bigcodebench_clone_dataset.xml.mlValidated``` is available in our repo under ```results/RQ2```

In [2]:
from src.config import *
PAIRS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml"
RESULTS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml.mlValidated"

In [3]:
with open(PAIRS, "r", encoding="utf-8") as f:
    xml_text = f.read()

num_clones = xml_text.count("</clone>")
print("Number of clone pairs:", num_clones)

Number of clone pairs: 56362


## Results

In [4]:
# Summary
from collections import Counter
 
counts = Counter()
total = 0

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines

        first_field = line.split(",", 1)[0].lower()
        if first_field in {"true", "false"}:
            counts[first_field] += 1
            total += 1

true_count = counts.get("true", 0)
false_count = counts.get("false", 0)

true_pct = (true_count / total * 100) if total > 0 else 0
false_pct = (false_count / total * 100) if total > 0 else 0

print(f"Total entries : {total}")
print(f"Detected clones : {true_count} ({true_pct:.2f}%)")
print(f"Not detected : {false_count} ({false_pct:.2f}%)")


Total entries : 56362
Detected clones : 51 (0.09%)
Not detected : 56311 (99.91%)


In [5]:
# Extract and print entries marked as true
true_entries = []

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        cols = [c.strip() for c in line.split(",")]

        if cols[0].lower() == "true" and len(cols) >= 4:
            true_entries.append(((cols[1], cols[2], cols[3]), (cols[6], cols[7], cols[8])))
 
print("Detected clones:")
for id1, id2 in true_entries:
    print(f"{id1}, {id2}")

print(f"\nTotal detected clones: {len(true_entries)}")

Detected clones:
("BigCodeBench/676_cot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/676_cot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']")
("BigCodeBench/33_zero-shot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/33_cot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']")
("BigCodeBench/33_cot gemma3:latest-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/33_cot llama3.1:latest-test 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/697_zero-shot gemma3:latest-complete 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/697_zero-shot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_4'", "'refac_5']")
("BigCodeBench/275_zero-shot llama3.1:latest-code 1 ['refac_2'", "'refac_4'", "'refac_6']"), (

In [11]:
import pandas as pd
import re
from collections import Counter


with open(RESULTS, "r") as f:
    lines = [l.strip() for l in f if l.strip()]

def parse_side(text):
    entry = re.search(r"(BigCodeBench/\d+)", text)
    strategy = re.search(r"_(cot|zero-shot)", text)
    model = re.search(r"(llama3\.1|deepseek-r1|gpt-oss|gemma3)", text)

    # >>> ADDED: context extraction
    context = re.search(r"-(test|ast|code|complete)\b", text)

    refacs = re.findall(r"refac_\d+", text)

    return {
        "entry": entry.group(1) if entry else None,
        "strategy": strategy.group(1) if strategy else None,
        "model": model.group(1) if model else None,
        "context": context.group(1) if context else None,   # <<< ADDED
        "refacs": refacs,
    }

records = []

for line in lines:
    parts = [p.strip() for p in line.split(",")]

    # Column 0 = TRUE / FALSE
    if parts[0].lower() != "true":
        continue

    # Left and right clone descriptions
    left_text = parts[1]
    right_text = parts[4]

    left = parse_side(left_text)
    right = parse_side(right_text)

    records.append({
        "entry": left["entry"],
        "strategy_left": left["strategy"],
        "strategy_right": right["strategy"],
        "model_left": left["model"],
        "model_right": right["model"],
        "context_left": left["context"],    # <<< ADDED
        "context_right": right["context"],  # <<< ADDED
        "refacs_left": left["refacs"],
        "refacs_right": right["refacs"],
    })

df = pd.DataFrame(records)

entry_counts = df["entry"].value_counts()

strategy_counts = Counter(df["strategy_left"]) + Counter(df["strategy_right"])
model_counts = Counter(df["model_left"]) + Counter(df["model_right"])

# >>> ADDED: context counter
context_counts = Counter(df["context_left"]) + Counter(df["context_right"])

refac_counts = Counter()
for _, r in df.iterrows():
    refac_counts.update(r["refacs_left"])
    refac_counts.update(r["refacs_right"])

strategy_pairs = Counter(
    zip(df["strategy_left"], df["strategy_right"])
)

print("\n=== Most problematic BigCodeBench entries ===")
print(entry_counts.head(10))

print("\n=== Strategy involvement ===")
for k, v in strategy_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Model involvement ===")
for k, v in model_counts.most_common():
    if k is None:
        continue
    print(f"{k:15s} {v}")

# >>> ADDED: context output
print("\n=== Context involvement ===")
for k, v in context_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Refactorings involved ===")
for k, v in refac_counts.most_common():
    print(f"{k:8s} {v}")

print("\n=== Strategy pairings ===")
for (s1, s2), v in strategy_pairs.most_common():
    print(f"{s1} vs {s2}: {v}")



=== Most problematic BigCodeBench entries ===
entry
BigCodeBench/911    10
BigCodeBench/546     2
BigCodeBench/122     2
BigCodeBench/4       2
BigCodeBench/33      2
BigCodeBench/667     2
BigCodeBench/697     1
BigCodeBench/293     1
BigCodeBench/676     1
BigCodeBench/275     1
Name: count, dtype: int64

=== Strategy involvement ===
zero-shot  33
cot        18

=== Model involvement ===
deepseek-r1     19
gemma3          15
gpt-oss         10
llama3.1        7

=== Context involvement ===
test       21
ast        11
complete   10
code       9

=== Refactorings involved ===
refac_1  39
refac_2  11
refac_3  1

=== Strategy pairings ===
zero-shot vs None: 33
cot vs None: 18


## Manual validation

In [2]:
import json
import random
import pandas as pd
from src.config import *

random.seed(42)

# ===== Load dataset =====
with open(FINAL_DATASET, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# ===== Collect all true clone pairs (original code + clones) =====
all_true_pairs = []

for entry in dataset:
    original_code = entry.get("code")
    clones = entry.get("clones", [])

    # Build a full group including original + clones
    group_codes = []

    if original_code:
        group_codes.append(original_code)

    group_codes.extend(
        clone["code"] for clone in clones if "code" in clone
    )

    # Need at least 2 elements to form pairs
    if len(group_codes) < 2:
        continue

    # Create pairs where original is always involved
    # (original vs each clone)
    for clone_code in group_codes[1:]:
        all_true_pairs.append((group_codes[0], clone_code, 1))

# ===== Compute 5% based on total number of clones (including original) =====
total_clone_instances = 0

for entry in dataset:
    original_present = 1 if entry.get("code") else 0
    clone_count = len(entry.get("clones", []))
    total_clone_instances += original_present + clone_count

if total_clone_instances == 0:
    raise ValueError("No clones/original code found in dataset.")

sample_size = max(1, int(total_clone_instances * 0.05))

print(f"Total clone instances (original + clones): {total_clone_instances}")
print(f"Sampling 5% = {sample_size} pairs")

# ===== Random sampling =====
if len(all_true_pairs) < sample_size:
    print(
        f"Warning: only {len(all_true_pairs)} valid original-clone pairs available."
    )
    sampled_pairs = all_true_pairs
else:
    sampled_pairs = random.sample(all_true_pairs, sample_size)

# ===== Create dataframe =====
df = pd.DataFrame(sampled_pairs, columns=["code_a", "code_b", "label"])

# ===== Save =====
output_path = "../results/RQ2/clone_inspection_sample.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Total sampled true clones:", len(df))
print("Labels:", df["label"].unique())

Total clone instances (original + clones): 7810
Sampling 5% = 390 pairs
Saved: ../results/RQ2/clone_inspection_sample.csv
Total sampled true clones: 390
Labels: [1]


### Labeling

In [1]:
!pip install ipywidgets pygments tqdm

In [10]:
import pandas as pd
import random
from IPython.display import display
import ipywidgets as widgets
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
from IPython.core.display import HTML

df = None
to_label_idx = None
state = {"i": 0}
user_col = None
output_path = "../results/RQ2/clone_inspection_sample.csv"

def load_dataset(user_id, start_index=None):
    global df, to_label_idx, user_col, state

    SEED = 42 if user_id == 1 else 10
    user_col = f"manual_label{user_id}"

     
    df = pd.read_csv(output_path)

    if "manual_label1" not in df.columns:
        df["manual_label1"] = pd.Series(dtype="Int64")
    else:
        df["manual_label1"] = df["manual_label1"].astype("Int64")

    if "manual_label2" not in df.columns:
        df["manual_label2"] = pd.Series(dtype="Int64")
    else:
        df["manual_label2"] = df["manual_label2"].astype("Int64")

    df = df[["code_a", "code_b", "label", "manual_label1", "manual_label2"]]

    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    to_label_idx = df.index.tolist()

    # skip already-labeled rows later during navigation
    if start_index is not None:
        state["i"] = max(0, min(start_index, len(to_label_idx)-1))
    else:
        state["i"] = 0

    print(f"User {user_id}")
    print(f"Column: {user_col}")
    print(f"Samples to label: {len(to_label_idx)}")
    print(f"Starting at position: {state['i']}")


def render_code(code):
    formatter = HtmlFormatter(style="friendly", full=True, noclasses=True)
    return HTML(highlight(str(code), PythonLexer(), formatter))


btn_t4 = widgets.Button(description="T4 Clone (4)", button_style="success")
btn_t3 = widgets.Button(description="T3 Clone (3)", button_style="info")
btn_t1 = widgets.Button(description="T1-T2 Clone (1)", button_style="warning")
btn_nonclone = widgets.Button(description="Non-Clone (0)", button_style="danger")

output = widgets.Output()


def show_sample():
    output.clear_output(wait=True)

    while state["i"] < len(to_label_idx):
        idx = to_label_idx[state["i"]]

        # skip already labeled rows
        if pd.notna(df.loc[idx, user_col]):
            state["i"] += 1
            continue

        row = df.loc[idx]

        with output:
            print(f"\nSample {state['i']+1}/{len(to_label_idx)} (index {idx})")
            print("=" * 80)

            print("\nCODE A:\n")
            display(render_code(row["code_a"]))

            print("\nCODE B:\n")
            display(render_code(row["code_b"]))

        return

    with output:
        print("Labeling complete!")

    df.to_csv(output_path, index=False)


def save_label(label):
    global df, user_col

    idx = to_label_idx[state["i"]]
    df.at[idx, user_col] = label
    df.to_csv(output_path, index=False)

    state["i"] += 1
    show_sample()


btn_t4.on_click(lambda _: save_label(4))
btn_t3.on_click(lambda _: save_label(3))
btn_t1.on_click(lambda _: save_label(1))
btn_nonclone.on_click(lambda _: save_label(0))

ui = widgets.VBox([
    widgets.HBox([ btn_nonclone,btn_t1,btn_t3, btn_t4]),
    output
])

USER_ID = 1
START_INDEX = 0   # change to resume later

display(ui)

load_dataset(user_id=USER_ID, start_index=START_INDEX)
show_sample()

User 1
Column: manual_label1
Samples to label: 390
Starting at position: 0


### Analysis

In [48]:
import pandas as pd

# Load dataset
path = "../results/RQ2/clone_inspection_sample.csv"
def load_and_analyze(path, user_col="manual_label1"):
    df = pd.read_csv(path)

    # Ensure numeric nullable integers
    for col in ["label", user_col]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # Keep only rows where user_col exists
    valid_df = df.dropna(subset=[user_col]).copy()

    print(f"Total labeled samples: {len(valid_df)}")

    # Compare user_col against ground-truth label
    matches = valid_df[
        valid_df[user_col] == valid_df["label"]
    ]

    non_matches = valid_df[
        valid_df[user_col] != valid_df["label"]
    ]

    print(f"\nMatching labels: {len(matches)}")
    print(f"Non-matching labels: {len(non_matches)}")

    # Distribution of matches
    print("\nMatching label distribution:")
    print(matches[user_col].value_counts().sort_index())

    # Print disagreements
    if len(non_matches) > 0:
        print("\nDisagreements:")
        print("-" * 80)

        for idx, row in non_matches.iterrows():
            print(f"\nRow index: {idx}")
            print(f"Ground truth label = {row['label']}")
            print(f"{user_col} = {row[user_col]}")

            print("\nCODE A:")
            print(str(row["code_a"])[:800])

            print("\nCODE B:")
            print(str(row["code_b"])[:800])

            print("\n" + "=" * 100)
    else:
        print("\nNo disagreements found.")

load_and_analyze(path, user_col="manual_label1")

Total labeled samples: 100

Matching labels: 96
Non-matching labels: 4

Matching label distribution:
manual_label1
0    50
1    46
Name: count, dtype: Int64

Disagreements:
--------------------------------------------------------------------------------

Row index: 12
Ground truth label = 1
manual_label1 = 0

CODE A:
import pandas as pd
PLANETS = ['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn',
    'Uranus', 'Neptune']
ELEMENTS = ['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']


def task_func():
    data = [[f'{p}:{e}' for e in ELEMENTS] for p in PLANETS]
    return pd.DataFrame(data, columns=ELEMENTS, index=PLANETS)


CODE B:
import pandas as pd
import random
from typing import List
PLANETS = ['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn',
    'Uranus', 'Neptune']
ELEMENTS = ['Hydrogen', 'Helium', 'Lithium', 'Beryllium', 'Boron', 'Carbon',
    'Nitrogen', 'Oxygen']


def task_func() ->pd.DataFrame:
    data = [[f'{p}:{e}' for e in ELEMENTS] for p in PLANETS